## imports

In [ ]:
! pip install ultralytics --quiet

In [ ]:
# train on the same v2_yolo_dataset 

from ultralytics import YOLO
model = YOLO("yolo26l.pt")  # load a pretrained model 


## Train

In [ ]:
model.train(
    data="data/v2_yolo_dataset/data.yaml",
    imgsz=640,
    batch=16,
    epochs=150,
    patience=10,  # Early stopping if val loss plateaus
    save_period=5,  # Save checkpoint every 5 epochs
    project="v3_model",
    device=0,  # GPU
    amp=True,  # Mixed precision (faster)
)

### continue if it stopped or crashed   

In [ ]:
model = YOLO("v3_model/weights/last.pt") 

# Resume training
results = model.train(
    data="data/v2_yolo_dataset/data.yaml",
    imgsz=640,
    batch=16,
    epochs=150,
    patience=10,
    save_period=5,
    project="v3_model",
    device=0,
    amp=True,
    exist_ok=True,
    resume=True,  # ← CRITICAL: Auto-resumes from checkpoint
)


## Accuracy / metrics


In [5]:
# Measure the accuracy of the model on the validation set for each class seperatly and compare it against v2 model
# v2 model results are in results/training/v2/ (files here)

import os
import pandas as pd

from ultralytics import YOLO

# Load the trained model
v3_model = YOLO("..\\models\\full\\best_v3.pt")  # Load the best v3 model
v2_model = YOLO("..\\models\\v2\\full\\best_v2.pt")  # Load the best v2 model

# Evaluate the v3 model on the validation set, and save the results in the results directory following the same structure as v2 model results
# don't forget they will be compared later on (especially the leoopard CLASS) so make sure to save the results in a way that they can be easily compared

v3_results = v3_model.val(data="..\\data\\v2_yolo_dataset\\data.yaml", save_dir="..\\results\\training\\v3\\")  # Save results in the same structure as v2 model results

Ultralytics 8.4.115  Python-3.13.9 torch-2.13.0+cpu CPU (Intel Core Ultra 7 155H)
YOLO26l summary (fused): 190 layers, 24,751,908 parameters, 0 gradients, 86.5 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 226.7151.2 MB/s, size: 336.0 KB)
val: Scanning C:\Users\Abdullah_Laptop\Desktop\KAUST SUMMER\Hunting-Leopards-with-Half-Brain\data\v2_yolo_dataset\valid\labels.cache... 1212 images, 58 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1212/1212 299.0Mit/s 0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 64, len(boxes) = 1406. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 76/76 15.7s/it 19:5419.5s
                   all       1212       1406      0.906      0.859      0.917      0.722
               leopard        849       

In [7]:
# compare the results of v2 and v3 models for each class separately, especially the leopard class
# using pandas to read the results and compare them

# 3rd: results\training\v3\results.csv
# 2nd: results\training\v2\results.csv



df_v3 = pd.read_csv("..\\results\\training\\v3\\results.csv")
df_v2 = pd.read_csv("..\\results\\training\\v2\\results.csv")

# Compare the results for each class
comparison = pd.merge(df_v2, df_v3, on="class", suffixes=("_v2", "_v3"))

# plot the comparison for each class, especially the leopard class
for class_name in comparison["class"].unique():
    class_comparison = comparison[comparison["class"] == class_name]
    print(f"Comparison for class: {class_name}")
    print(class_comparison[["precision_v2", "precision_v3", "recall_v2", "recall_v3", "mAP50_v2", "mAP50_v3"]])
    print("\n")



FileNotFoundError: [Errno 2] No such file or directory: '..\\results\\training\\v2\\results.csv'

## Export

In [ ]:
! pip install onnx onnxruntime --quiet

model.export(format="onnx", imgsz=640)
print("✓ Exports complete")